# ZK-KGVerify: Privacy-Preserving Verification of Knowledge Graph Reasoning using Zero-Knowledge Proofs and Blockchain

**Full experimental pipeline for the paper.**

Pipeline:
1. Load FB15k-237 knowledge graph dataset
2. Train 3 KG embedding models (TransE, RotatE, CompGCN) — 200 epochs
3. Evaluate link prediction on full test set (MRR, Hits@1/3/10)
4. Generate Zero-Knowledge Proofs for model predictions
5. Store verification results on simulated blockchain
6. Generate paper figures, tables, and analysis

**Runtime: ~2-3 hours on Colab T4 GPU (200 epochs, full test set evaluation)**

## Section 1: Setup & Install Dependencies

In [ ]:
# ============================================================
# STEP 1: Mount Google Drive & navigate to project folder
# ============================================================
# INSTRUCTIONS:
#   1. Upload the entire "new-blockchain-paper" folder to your Google Drive
#   2. Update PROJECT_PATH below if you placed it somewhere else
# ============================================================

import os

# --- Change this if your folder is in a different Drive location ---
PROJECT_PATH = "/content/drive/MyDrive/new-blockchain-paper"

from google.colab import drive
drive.mount('/content/drive')

os.chdir(PROJECT_PATH)
print(f"Working directory: {os.getcwd()}")
print(f"Contents: {os.listdir('.')}")

In [ ]:
# ============================================================
# STEP 2: Install dependencies & verify project structure
# ============================================================
!pip install matplotlib pandas numpy seaborn tqdm scikit-learn -q

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Verify project files exist
import sys
sys.path.insert(0, '.')

required = ['src', 'configs', 'src/__init__.py', 'configs/__init__.py',
            'src/data_loader.py', 'src/models.py', 'src/trainer.py',
            'src/zkp_module.py', 'src/blockchain_module.py',
            'src/visualization.py', 'configs/config.py']
missing = [f for f in required if not os.path.exists(f)]
if missing:
    print(f"\n❌ MISSING FILES: {missing}")
    print(f"   Current dir: {os.getcwd()}")
    print(f"   Fix PROJECT_PATH in the cell above.")
else:
    print(f"\n✅ All project files found!")

# Ensure output directories exist
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

## Section 2: Load FB15k-237 Dataset

In [ ]:
import sys
sys.path.insert(0, '.')

from src.data_loader import FB15k237Dataset, get_data_loaders
from configs.config import (
    BATCH_SIZE, NEGATIVE_SAMPLE_SIZE, EMBEDDING_DIM, MARGIN,
    NUM_EPOCHS, LEARNING_RATE, MODELS, DEVICE, NUM_ZKP_SAMPLES,
    RESULTS_DIR, EVAL_BATCH_SIZE, EVAL_MAX
)

# Load dataset
dataset = FB15k237Dataset(data_dir='./data')

print(f"\nDataset Summary:")
print(f"  Entities:  {dataset.num_entities:,}")
print(f"  Relations: {dataset.num_relations}")
print(f"  Train:     {len(dataset.train_triples):,} triples")
print(f"  Valid:     {len(dataset.valid_triples):,} triples")
print(f"  Test:      {len(dataset.test_triples):,} triples")
print(f"\nConfig: {NUM_EPOCHS} epochs, {EMBEDDING_DIM}-dim, eval_max={EVAL_MAX}")

# Create data loader
train_loader = get_data_loaders(dataset, batch_size=BATCH_SIZE, negative_sample_size=NEGATIVE_SAMPLE_SIZE)

## Section 3: Train KG Embedding Models

In [ ]:
import time
import torch
from src.models import get_model
from src.trainer import train_model, evaluate_model
import configs.config as config_module

all_histories = {}
all_metrics = {}
trained_models = {}

CHECKPOINT_DIR = './checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

for model_name in MODELS:
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'{model_name}_trained.pt')
    
    # --- Resume from checkpoint if available ---
    if os.path.exists(ckpt_path):
        print(f"\n{'='*50}")
        print(f"LOADING CHECKPOINT: {model_name}")
        print(f"{'='*50}")
        
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        
        model = get_model(
            model_name,
            num_entities=dataset.num_entities,
            num_relations=dataset.num_relations,
            embedding_dim=EMBEDDING_DIM,
            margin=MARGIN
        )
        model.load_state_dict(ckpt['model_state'])
        model = model.to(DEVICE)
        
        all_histories[model_name] = ckpt['history']
        all_metrics[model_name] = ckpt['metrics']
        trained_models[model_name] = model
        
        print(f"  Loaded! MRR={ckpt['metrics']['MRR']:.4f}")
        continue
    
    # --- Train from scratch ---
    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    print(f"{'='*50}")
    
    try:
        model = get_model(
            model_name,
            num_entities=dataset.num_entities,
            num_relations=dataset.num_relations,
            embedding_dim=EMBEDDING_DIM,
            margin=MARGIN
        )
        
        history = train_model(model, train_loader, dataset, config_module, device=DEVICE)
        all_histories[model_name] = history
        
        print(f"\n  Evaluating {model_name} on full test set...")
        eval_max = getattr(config_module, 'EVAL_MAX', None)
        metrics = evaluate_model(model, dataset, config_module, device=DEVICE, max_eval=eval_max)
        all_metrics[model_name] = metrics
        trained_models[model_name] = model
        
        # --- Save checkpoint immediately ---
        torch.save({
            'model_state': model.state_dict(),
            'history': history,
            'metrics': metrics,
        }, ckpt_path)
        print(f"  Checkpoint saved: {ckpt_path}")
    
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\n  Skipped {model_name} — CUDA out of memory")
            torch.cuda.empty_cache()
        else:
            raise

print(f"\nSuccessfully trained: {list(trained_models.keys())}")

## Section 4: Evaluate Link Prediction (Results Table)

In [ ]:
import pandas as pd

# Create results table
results_df = pd.DataFrame(all_metrics).T
results_df = results_df[['MRR', 'Hits@1', 'Hits@3', 'Hits@10', 'Mean_Rank', 'eval_time']]
results_df = results_df.round(4)

print("\nLink Prediction Results on FB15k-237")
print("=" * 70)
print(results_df.to_string())
print("=" * 70)

# Training time summary
print("\nTraining Time Summary:")
for model_name, history in all_histories.items():
    print(f"  {model_name}: {history['total_time']:.2f}s")

## Section 5: Zero-Knowledge Proof Module

In [ ]:
import numpy as np
from src.zkp_module import (
    generate_proof, verify_proof, batch_generate_proofs,
    batch_verify_proofs, tamper_proof
)

# Select best model
best_model_name = max(all_metrics, key=lambda k: all_metrics[k]['MRR'])
best_model = trained_models[best_model_name]
print(f"Using best model: {best_model_name} (MRR={all_metrics[best_model_name]['MRR']:.4f})")

best_model.eval()
best_model = best_model.to(DEVICE)

# Set graph for GCN models
if hasattr(best_model, 'set_graph'):
    edge_index = dataset.train_triples[:, [0, 2]].t().to(DEVICE)
    edge_type = dataset.train_triples[:, 1].to(DEVICE)
    best_model.set_graph(edge_index, edge_type)

# Sample triples
num_samples = min(NUM_ZKP_SAMPLES, len(dataset.test_triples))
sample_indices = torch.randperm(len(dataset.test_triples))[:num_samples]
sample_triples = dataset.test_triples[sample_indices]

embedding_vectors = []
prediction_scores = []
triples_list = []

with torch.no_grad():
    for i in range(num_samples):
        h, r, t = sample_triples[i].tolist()
        h_idx = torch.tensor([h], device=DEVICE)
        r_idx = torch.tensor([r], device=DEVICE)
        t_idx = torch.tensor([t], device=DEVICE)
        
        emb_vec = best_model.get_embedding_vector(h_idx, r_idx, t_idx)
        embedding_vectors.append(emb_vec.cpu().numpy().flatten())
        
        scores = best_model.predict(h_idx, r_idx)
        prediction_scores.append(scores[0, t].item())
        triples_list.append((h, r, t))

print(f"\nPrepared {num_samples} predictions for ZKP generation")

In [ ]:
# Generate proofs
print(f"Generating {num_samples} ZK proofs...")
proofs, gen_stats = batch_generate_proofs(
    embedding_vectors, prediction_scores, triples_list, best_model_name
)

print(f"\nProof Generation Statistics:")
print(f"  Total time:  {gen_stats['total_gen_time']:.2f}s")
print(f"  Avg time:    {gen_stats['avg_gen_time']*1000:.2f} ms")
print(f"  Avg size:    {gen_stats['avg_proof_size_bytes']:.0f} bytes")

# Verify proofs
print(f"\nVerifying {num_samples} proofs...")
results, verify_stats = batch_verify_proofs(proofs)

print(f"\nVerification Statistics:")
print(f"  Valid:   {verify_stats['num_valid']}/{verify_stats['num_verified']}")
print(f"  Rate:    {verify_stats['verification_rate']*100:.1f}%")
print(f"  Avg time: {verify_stats['avg_verify_time']*1000:.2f} ms")

# Tamper detection
print(f"\nTesting tamper detection on 100 proofs...")
tampered = [tamper_proof(p) for p in proofs[:100]]
tampered_results, tampered_stats = batch_verify_proofs(tampered)
print(f"  Tampered detected: {tampered_stats['num_invalid']}/100 ({tampered_stats['num_invalid']}%)")

## Section 6: Blockchain Module

In [ ]:
from src.blockchain_module import create_blockchain

blockchain = create_blockchain(mode='local')

print(f"Logging {num_samples} verification records to blockchain...")
bc_start = time.time()

for i, proof in enumerate(proofs):
    blockchain.add_verification_record(
        model_id=proof.model_id,
        triple=proof.triple,
        prediction_score=proof.score,
        zkp_commitment=proof.commitment,
        zkp_verified=results[i],
        proof_hash=proof.prediction_hash
    )

blockchain.mine_pending()
bc_time = time.time() - bc_start
bc_stats = blockchain.get_stats()

print(f"\nBlockchain Statistics:")
print(f"  Time:         {bc_time:.2f}s")
print(f"  Blocks:       {bc_stats['num_blocks']}")
print(f"  Transactions: {bc_stats['total_transactions']}")
print(f"  Total gas:    {bc_stats['total_gas_used']:,}")
print(f"  Avg gas/tx:   {bc_stats['avg_gas_per_tx']:,.0f}")
print(f"  Chain valid:  {bc_stats['chain_valid']}")

## Section 7: End-to-End Pipeline Summary

In [ ]:
# Collect detailed ZKP stats for plots
gen_times = []
verify_times = []
proof_sizes = [p.proof_size_bytes for p in proofs]

for i in range(min(200, len(proofs))):
    start = time.time()
    generate_proof(embedding_vectors[i], prediction_scores[i], triples_list[i], best_model_name)
    gen_times.append(time.time() - start)
    
    start = time.time()
    verify_proof(proofs[i])
    verify_times.append(time.time() - start)

zkp_full_stats = {
    **gen_stats,
    **verify_stats,
    'gen_times': gen_times,
    'verify_times': verify_times,
    'proof_sizes': proof_sizes,
    'tamper_detection_rate': tampered_stats['num_invalid'] / 100,
}

n_models = len(trained_models)
pipeline_times = {
    f'Training ({n_models} models)': sum(h['total_time'] for h in all_histories.values()),
    'Evaluation': sum(m['eval_time'] for m in all_metrics.values()),
    'ZKP Generation': gen_stats['total_gen_time'],
    'ZKP Verification': verify_stats['total_verify_time'],
    'Blockchain Storage': bc_time,
}

print("Pipeline Timing:")
for stage, t in pipeline_times.items():
    print(f"  {stage}: {t:.2f}s")
print(f"  Total: {sum(pipeline_times.values()):.2f}s")

## Section 8: Generate Figures for Paper

In [ ]:
from src.visualization import (
    plot_training_curves, plot_metrics_comparison, plot_zkp_overhead,
    plot_blockchain_stats, plot_end_to_end_pipeline, generate_latex_tables,
    save_all_results
)

save_dir = './results'

print("Generating figures...")
plot_training_curves(all_histories, save_dir=save_dir)
plot_metrics_comparison(all_metrics, save_dir=save_dir)
plot_zkp_overhead(zkp_full_stats, save_dir=save_dir)
plot_blockchain_stats(bc_stats, save_dir=save_dir)
plot_end_to_end_pipeline(pipeline_times, save_dir=save_dir)

print("\nGenerating LaTeX tables...")
latex = generate_latex_tables(all_metrics, zkp_full_stats, bc_stats, save_dir=save_dir)

print("\nSaving all results...")
save_all_results(all_metrics, all_histories, zkp_full_stats, bc_stats, pipeline_times, save_dir=save_dir)

print("\nDone! All files in ./results/")

## Section 9: Display Figures

In [ ]:
from IPython.display import Image, display

figures = [
    ('Training Loss Curves', 'training_curves.png'),
    ('Link Prediction Metrics', 'metrics_comparison.png'),
    ('ZKP Overhead Analysis', 'zkp_overhead.png'),
    ('Blockchain Statistics', 'blockchain_stats.png'),
    ('Pipeline Timing', 'pipeline_timing.png'),
]

for title, fname in figures:
    fpath = f'./results/{fname}'
    if os.path.exists(fpath):
        print(f"\n{'='*50}")
        print(f"  {title}")
        print(f"{'='*50}")
        display(Image(filename=fpath, width=800))

In [ ]:
# Display LaTeX tables
print("\nLaTeX Tables for Paper:")
print("=" * 50)
with open('./results/tables.tex', 'r') as f:
    print(f.read())

In [ ]:
# Final summary
print("\n" + "=" * 70)
print("  ZK-KGVerify — EXPERIMENT COMPLETE")
print("=" * 70)
print(f"\n  Best model: {best_model_name} (MRR={all_metrics[best_model_name]['MRR']:.4f})")
print(f"  ZKP overhead: {zkp_full_stats['avg_gen_time']*1000:.2f}ms per proof")
print(f"  Tamper detection: {zkp_full_stats['tamper_detection_rate']*100:.1f}%")
print(f"  Blockchain cost: {bc_stats['avg_gas_per_tx']:,.0f} gas/tx")
print(f"\n  All results saved in ./results/")
print("=" * 70)